# Prepare LLM Annotation Batches

This notebook creates auditable JSONL inputs for the locked Codex annotator workflow and independently validates the resulting canonical outputs. Actual model execution is handled by `run_codex_annotation.py`, which launches one fresh, read-only `codex exec` run per batch.

In [4]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "01_classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CLASSIFICATION_DIR = PROJECT_ROOT / "data/interim/lsc/classification"
LLM_DIR = CLASSIFICATION_DIR / "llm_annotation"
CODEX_PRODUCTION_DIR = LLM_DIR / "codex/production"
BATCH_DIR = CODEX_PRODUCTION_DIR / "inputs"
RAW_OUTPUT_DIR = CODEX_PRODUCTION_DIR / "outputs"
PARSED_DIR = CODEX_PRODUCTION_DIR
RUN_DIR = CODEX_PRODUCTION_DIR / "runs"

for path in [BATCH_DIR, RAW_OUTPUT_DIR, PARSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LLM_POOL_PATH = CODEX_PRODUCTION_DIR / "training_pool_with_annotation_ids.csv"
BATCH_MANIFEST_PATH = CODEX_PRODUCTION_DIR / "manifest.csv"
PROMPT_PATH = PROJECT_ROOT / "notebooks/01_classification/prompts/annotator_v4.md"
CODEBOOK_PATH = PROJECT_ROOT / "notebooks/01_classification/codebooks/codebook_v4.md"
RUNNER_PATH = PROJECT_ROOT / "notebooks/01_classification/run_codex_annotation.py"
BATCH_SIZE = 75
PROMPT_VERSION = "annotator_v4"
CODEBOOK_VERSION = "v0.4"

## Inspect Annotator Input Batches

The append-only input batches are prepared by `run_codex_annotation.py`. This notebook verifies their coverage and checksums without regenerating or overwriting completed batches. Use the runner's `prepare-production-extension` command to add further annotation tranches.

In [5]:
pool = pd.read_csv(LLM_POOL_PATH)
manifest = pd.read_csv(BATCH_MANIFEST_PATH)

def file_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


batch_paths = [BATCH_DIR / name for name in manifest["batch_name"]]
missing_batches = [path.name for path in batch_paths if not path.exists()]
if missing_batches:
    raise FileNotFoundError(f"Missing input batches: {missing_batches}")
checksum_mismatches = [
    path.name
    for path, expected_sha in zip(batch_paths, manifest["sha256"])
    if file_sha256(path) != expected_sha
]
if checksum_mismatches:
    raise ValueError(f"Batch checksum mismatches: {checksum_mismatches}")
if manifest["rows"].sum() != len(pool):
    raise ValueError("Batch-manifest row count differs from the production pool.")
if pool["annotation_id"].duplicated().any() or pool["context_id"].duplicated().any():
    raise ValueError("Production pool contains duplicate annotation or context IDs.")

print(f"Prompt: {PROMPT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Codebook: {CODEBOOK_PATH.relative_to(PROJECT_ROOT)}")
print(f"Runner: {RUNNER_PATH.relative_to(PROJECT_ROOT)}")
print(f"Verified {len(manifest):,} batches containing {len(pool):,} unique rows.")

Prompt: notebooks/01_classification/prompts/annotator_v4.md
Codebook: notebooks/01_classification/codebooks/codebook_v4.md
Runner: notebooks/01_classification/run_codex_annotation.py
Verified 41 batches containing 3,000 unique rows.


## Parse Annotator Outputs

The runner writes one schema- and hierarchy-validated canonical JSONL output per batch to `codex/production/outputs/`. This section independently requires complete batch coverage, verifies exact IDs and order against each input, derives frame labels, and reports distributions before combining all current production labels.

In [6]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            text = line.strip()
            if not text:
                continue
            try:
                rows.append(json.loads(text))
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON in {path.name} line {line_number}: {error}") from error
    return rows

required_output_columns = [
    "annotation_id",
    "substantive_target_discourse",
    "clinical_frame_present",
    "lived_experience_frame_present",
    "confidence",
]
confidence_values = {"high", "medium", "low"}
input_files = sorted(BATCH_DIR.glob("*.jsonl"))
raw_files = sorted(RAW_OUTPUT_DIR.glob("*.jsonl"))
if not raw_files:
    print(f"No raw annotator outputs found in {RAW_OUTPUT_DIR.relative_to(PROJECT_ROOT)} yet.")
else:
    expected_names = {path.name for path in input_files}
    actual_names = {path.name for path in raw_files}
    if actual_names != expected_names:
        raise ValueError(
            f"Incomplete or unexpected outputs: missing={sorted(expected_names - actual_names)}, "
            f"extra={sorted(actual_names - expected_names)}"
        )
    parsed_rows = []
    for input_path, output_path in zip(input_files, raw_files):
        input_rows = read_jsonl(input_path)
        output_rows = read_jsonl(output_path)
        input_ids = [row["annotation_id"] for row in input_rows]
        output_ids = [row.get("annotation_id") for row in output_rows]
        if output_ids != input_ids:
            raise ValueError(f"ID coverage/order mismatch: {output_path.name}")
        for row in output_rows:
            if set(row) != set(required_output_columns):
                raise ValueError(f"Unexpected output fields in {output_path.name}: {sorted(row)}")
            substantive = row["substantive_target_discourse"]
            clinical = row["clinical_frame_present"]
            lived = row["lived_experience_frame_present"]
            if type(substantive) is not bool:
                raise ValueError(f"Invalid Stage-0 value: {row['annotation_id']}")
            if substantive and (type(clinical) is not bool or type(lived) is not bool):
                raise ValueError(f"Substantive row requires boolean Stage-1 labels: {row['annotation_id']}")
            if not substantive and (clinical is not None or lived is not None):
                raise ValueError(f"Non-substantive row requires null Stage-1 labels: {row['annotation_id']}")
            if row["confidence"] not in confidence_values:
                raise ValueError(f"Invalid confidence: {row['annotation_id']}")
        for row in output_rows:
            row = row.copy()
            row["source_file"] = output_path.name
            parsed_rows.append(row)
    parsed = pd.DataFrame(parsed_rows)
    missing = sorted(set(required_output_columns) - set(parsed.columns))
    if missing:
        raise ValueError(f"LLM output is missing required columns: {missing}")
    if parsed["annotation_id"].duplicated().any() or len(parsed) != len(pool):
        raise ValueError(f"Expected {len(pool):,} unique annotations, found {len(parsed):,} rows.")
    parsed["derived_frame"] = [
        "non_substantive_or_insufficient"
        if not substantive
        else "mixed"
        if clinical and lived
        else "clinical_only"
        if clinical
        else "lived_only"
        if lived
        else "substantive_other"
        for substantive, clinical, lived in zip(
            parsed["substantive_target_discourse"],
            parsed["clinical_frame_present"],
            parsed["lived_experience_frame_present"],
        )
    ]
    parsed_path = PARSED_DIR / "labels.csv"
    parsed.to_csv(parsed_path, index=False)
    print(f"Parsed {len(parsed):,} LLM annotation rows to {parsed_path.relative_to(PROJECT_ROOT)}")
    display(parsed.groupby(["derived_frame", "confidence"]).size().rename("rows").reset_index())

    metadata_rows = []
    for metadata_path in sorted(RUN_DIR.glob("*/*/attempt_*_metadata.json")):
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata.get("status") == "valid":
            metadata_rows.append(
                {
                    "batch_name": metadata["batch_name"],
                    "reasoning_effort": metadata["reasoning_effort"],
                    **metadata.get("token_usage", {}),
                }
            )
    if metadata_rows:
        token_usage = pd.DataFrame(metadata_rows)
        display(token_usage)
        display(token_usage.select_dtypes("number").sum().rename("total"))


Parsed 3,000 LLM annotation rows to data/interim/lsc/classification/llm_annotation/codex/production/labels.csv


,derived_frame,confidence,rows
0,clinical_only,high,1016
1,clinical_only,medium,210
2,lived_only,high,394
3,lived_only,medium,150
4,mixed,high,99
5,mixed,medium,100
6,non_substantive_or_insufficient,high,838
7,non_substantive_or_insufficient,medium,170
8,substantive_other,high,8
9,substantive_other,medium,15


,batch_name,reasoning_effort,input_tokens,cached_input_tokens,output_tokens,reasoning_output_tokens
0,annotator_batch_001.jsonl,high,101301,59904,5437,2148
1,annotator_batch_002.jsonl,high,108808,88576,6024,2716
2,annotator_batch_003.jsonl,high,99707,59392,5861,2553
3,annotator_batch_004.jsonl,high,108691,64512,6012,2595
4,annotator_batch_005.jsonl,high,100635,58368,5654,2312
5,annotator_batch_006.jsonl,high,113871,64000,6426,2940
6,annotator_batch_007.jsonl,high,107705,88576,6554,3261
7,annotator_batch_008.jsonl,high,100211,78336,5202,1864
8,annotator_batch_009.jsonl,high,107100,88576,5788,2479
9,annotator_batch_010.jsonl,high,111125,88576,6722,3380


input_tokens               4280245
cached_input_tokens        3157504
output_tokens               253488
reasoning_output_tokens     118816
Name: total, dtype: int64